<a href="https://colab.research.google.com/github/Wezz-git/AI-samples/blob/main/(DQA)_Data_Validation_and_Resilience.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project**: Automated Data Quality Assurance (DQA).

**Goal**: Implement automated, defensive checks that guarantee the data entering a live model is valid. The trained models expect clean data (no negative ages, no text in the balance column). This project enforces those rules.

**Skill**: Data Quality Assurance (DQA) and using Python's fundamental assertion statements and data type checks to create a production guardrail.

Assertion: A command that states a condition must be true. If the condition is false, the program immediately stops and throws an error, preventing the bad data from ever reaching the model.

If a corrupted CSV gives you a negative age (-10), your model will output a nonsensical prediction. An assertion stops the entire pipeline before the model runs, saving the company from making a bad decision.

Setup & Mock Data Creation

In [5]:
import pandas as pd
import numpy as np

# Create clean DF (expected input)
data = {
    'Tenure': [30, 40, 50, 60, 70],
    'MonthlyCharges': [55.90, 89.98, 105.00, 22.00, 48.80],
    'age': [25, 35, 45, 55, 65],
    'Contract_Month_to_Month': [1, 0, 0, 1, 0],
    'Prediction_Target': [0, 1, 0, 1, 0]
}

df_cleaned = pd.DataFrame(data)

# Create a corrupted DF (Failed Output)
data_corrupted = {
    'Tenure': [30, 40, 50, 60, 70],
    'MonthlyCharges': [55.90, '89.98USD', 105.00, 22.00, 48.80],    # Type Error
    'age': [25, 35, 45, -18, 65],                                 # Logial Error (Negative age)
    'Contract_Month_to_Month': [1, 0, 0, 1, 0],
    'Prediction_Target': [0, 1, 0, 1, 0]
}

df_corrupted = pd.DataFrame(data_corrupted)

print("Mock data created. df_clean is safe. df_corrupt has errors.")

Mock data created. df_clean is safe. df_corrupt has errors.


Implemented Data Quality Assertation (DQA)

In [6]:
# Define validation Function
def validate_data_for_prediction(df_input: pd.DataFrame, min_age=24):
  # Checks to stop wrong data from reaching model
  print(f" Running validation on {len(df_input)} samples")

  # Checks for unexpected non-numeric types in key columns
  # Model expects all features to be numeric

  for col in ['Tenure', 'MonthlyCharges', 'age']:

    # Checks dtype is an object (text, not numeric)
    assert df_input[col].dtype != object, \
      f"Validation Error: Column '{col}' contains text (dtype: {df_input[col].dtype}). Must be numeric."

  # Checks logical boundaries (domain knowledge)
  # 'age' cannot be negative
  assert (df_input['age'] >= min_age).all(), \
    f"Validation Error: 'age' cannot be less than {min_age}"

  # Checks for structural integrity (Basic check)
  # No critical columns should be missing (all NaN)
  assert df_input.isnull().all().sum() == 0, \
    "Validation Error: Missing values found in critical column."

  print("Data validation success.")
  return True

print("Test Clean data.")
print(validate_data_for_prediction(df_cleaned))

Test Clean data.
 Running validation on 5 samples
Data validation success.
True


Test the Crash (prove resillience)

Want the system to crash here instead of feeding bad data to your trained XGBoost model. Should immediately crash and show the AssertionError message for the negative age or the text in MonthlyCharges.

In [8]:
# Test corrupted data (should fail/crash)
print("\nTest corrupted data.")

try:
  validate_data_for_prediction(df_corrupted)
except AssertionError as e:
  print(e)
  print("System prevented bad data from reaching model.")


Test corrupted data.
 Running validation on 5 samples
Validation Error: Column 'MonthlyCharges' contains text (dtype: object). Must be numeric.
System prevented bad data from reaching model.
